# Phase 6, track M (Mirul): M2, flip test-time augmentation (T4, no training)

**What this notebook does, in plain words:** it takes Kimee's two best trained models (`x101_2048_m28` and
`r50_2048`, the members of our best ensembles) and lets them look at every val and test image **three more times:
mirrored left-right (`h`), upside-down (`v`) and both (`hv`)**. Each prediction is flipped back, so all masks line up
with the original image. Later, on the laptop, the 4 views per model are merged like an ensemble: a filament found in
most views gets a high score, one found in a single view a low score. That usually removes doubtful masks and keeps
good ones ("test-time augmentation", TTA). No training happens, so this is cheap.

| Step | What | Time on T4 (estimate) |
|---|---|---|
| 0 | probe: 7 images with one flip; shows memory and speed | ~5 min after install |
| 1 | 2 models × 3 flips × 321 images (141 val + 180 test) | ~1–2 h (the probe prints a better estimate) |

The un-flipped predictions already exist in git (`Result Step 5/Step 2/` and `Result Phase 4/`), so they are not
made again.

**Where things go:** Kimee's checkpoints are only **read**. Prediction files are written to the team Drive
**`solar_filament/Phase 6/Mirul/`** only.

**How to run:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all. Keep the tab open.
**If Colab disconnects:** run all again; finished files are skipped.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Mount Google Drive (team folder)
Needs the shortcut to Kimee's `solar_filament` folder in your My Drive (`HANDOVER.md` §0.6).

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/solar_filament"
MODELS = {   # name -> flags it was trained with (must match for prediction)
    "x101_2048_m28": "--backbone x101 --resolution 2048 --mask-res 28",
    "r50_2048": "--resolution 2048",
}
CKPT = {n: f"{DRIVE_DIR}/output/detectron2/{n}/model_final.pth" for n in MODELS}
for n, c in CKPT.items():
    assert os.path.exists(c), f"Checkpoint not found: {c} (add the team Drive shortcut, HANDOVER.md section 0.6)"
MY_DIR = f"{DRIVE_DIR}/Phase 6/Mirul"        # the ONLY Drive place this notebook writes to
os.makedirs(MY_DIR, exist_ok=True)
OUT_DIR = "/content/out"                     # local scratch; files are copied to MY_DIR
os.environ["SOLAR_OUTPUT_DIR"] = OUT_DIR
print("Reading checkpoints from Kimee's folder, writing to:", MY_DIR)

## 2. Get the latest code (pulls if the repo is already cloned)

In [ ]:
BRANCH = "Dev"
REPO = "github.com/amiruliman592/Kaggle-Competition-.git"
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = f"https://{token}@{REPO}" if token else f"https://{REPO}"

if os.path.exists("/content/solar"):
    !git -C /content/solar pull -q origin {BRANCH}
else:
    !git clone -q --depth 1 -b {BRANCH} {url} /content/solar
%cd /content/solar
os.environ["SOLAR_ROOT"] = "/content/solar"
!git log --oneline -1

## 3. Install dependencies (Detectron2 build ~5–8 min)

In [ ]:
!pip install -q -r requirements.txt
!pip install -q --no-build-isolation 'git+https://github.com/facebookresearch/detectron2.git'
import torch, detectron2
print("torch", torch.__version__, "| detectron2", detectron2.__version__, "| GPU", torch.cuda.get_device_name(0))

## 4. Quick checks (~30 s) and helpers

In [ ]:
import shutil, subprocess, time

def sh(cmd):
    """Run one command (output shown live); stop the notebook if it fails."""
    t0 = time.time()
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"Command failed (exit {p.returncode}): {cmd}")
    print(f"--- done in {(time.time() - t0) / 60:.1f} min")

def dump_flip(name, flip, probe=False):
    """Predict val + test with one flip; copy the two files to MY_DIR. Skips files that already exist."""
    tag = f"{name}_flip{flip}"
    if not probe and all(os.path.exists(f"{MY_DIR}/raw_preds_{tag}_{s}.json.gz") for s in ("val", "test")):
        print(f"{tag}: already done, skipped.")
        return
    run_name = f"probe_{name}" if probe else name
    sh(f"python run_pipeline.py --step dump --name {run_name} {MODELS[name]} --weights '{CKPT[name]}' --flip {flip}"
       + (" --dry-run" if probe else ""))
    if not probe:
        for s in ("val", "test"):
            shutil.copy(f"{OUT_DIR}/raw_preds_{tag}_{s}.json.gz", MY_DIR)
        print(f"{tag}: saved to {MY_DIR}")

sh("python -m pytest -q -p no:warnings tests/test_pipeline_flags.py tests/test_flip.py tests/test_official_metric.py")

## Step 0: probe (7 images, larger model, one flip)
X101 at 2048 has only been run on an A100 before. Look at `Peak GPU memory` (must be below ~14 GB) and `s/img`.
Full run ≈ s/img × 321 images × 6 (r50 is a bit faster). If this fails with "CUDA out of memory", stop and tell Claude.

In [ ]:
dump_flip("x101_2048_m28", "h", probe=True)

## Step 1: all flips for both models (~1–2 h)

In [ ]:
for name in MODELS:
    for flip in ("h", "v", "hv"):
        dump_flip(name, flip)
print("All flips done.")

## 5. Send results back
Download the 12 files (also in the team Drive `Phase 6/Mirul/`) into the repo folder `Result Phase 6/Mirul/` on
your laptop, then tell Claude. On the laptop, Claude fuses each model's 4 views with `src/ensemble.py` (the original
predictions come from `Result Step 5/Step 2/raw_preds_x101_2048_m28_*.json.gz` and
`Result Phase 4/raw_preds_r50_2048_*.json.gz`), then the two models together, and scores them against exp010 / exp013
with your M1 bootstrap. Also copy the `Peak GPU memory` and `[Dump] ... s/img` lines.
Then **disconnect the runtime** (Runtime → Disconnect and delete runtime).

In [ ]:
from google.colab import files
for name in MODELS:
    for flip in ("h", "v", "hv"):
        for s in ("val", "test"):
            f = f"{MY_DIR}/raw_preds_{name}_flip{flip}_{s}.json.gz"
            if os.path.exists(f):
                files.download(f)
            else:
                print("not found (not finished?):", f)